# R3c MoE Bridge — CPU engineering only
Run All once with accelerator **None**. This release is not a GPU training release. Uses installed PyTorch and NumPy; no tokens or secret setup required. Saves local synthetic checkpoint and aggregate reports. Public code submission/CI are separate from Kaggle Output persistence.


In [ ]:
from pathlib import Path
import os, subprocess, sys
SOURCE_SHA='0b04e89ef6f14f7c9ad3aa6da5692df8daef9b9b'
ROOT=Path('/kaggle/working/r3c-bridge-code-'+SOURCE_SHA[:12])
OUTPUT=Path('/kaggle/working/r3c-bridge-cpu-'+SOURCE_SHA[:12])
def run(*args, **kwargs):
    print('[R3c CPU]', *map(str,args), flush=True)
    return subprocess.run(list(map(str,args)),check=True,**kwargs)
ROOT.mkdir(parents=True, exist_ok=True)
run('git','init','--quiet',ROOT)
remote=subprocess.run(['git','-C',str(ROOT),'remote','get-url','origin'],capture_output=True,text=True)
if remote.returncode:
    run('git','-C',ROOT,'remote','add','origin','https://github.com/ArcheLabs/mini-cells.git')
run('git','-C',ROOT,'config','remote.origin.promisor','true')
run('git','-C',ROOT,'config','remote.origin.partialclonefilter','blob:none')
run('git','-C',ROOT,'sparse-checkout','init','--no-cone')
run('git','-C',ROOT,'sparse-checkout','set','--no-cone',*['/pyproject.toml', '/src/', '/shared/python/', '/research/projects/cellular-language-models/engineering/kaggle-continuation/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/', '/research/projects/cellular-language-models/experiments/006-path-first-clm/', '/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/implementation/', '/research/projects/continual-learning/context/04-continual-learning/scripts/research/', '/research/projects/cellular-language-models/context/08-hybrid-clm/scripts/research/'])
head=subprocess.run(['git','-C',str(ROOT),'rev-parse','HEAD'],capture_output=True,text=True)
if head.returncode or head.stdout.strip()!=SOURCE_SHA:
    run('git','-C',ROOT,'fetch','--depth=1','--filter=blob:none','origin',SOURCE_SHA)
    run('git','-C',ROOT,'checkout','--detach','FETCH_HEAD')
actual=subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip()
if actual!=SOURCE_SHA:raise RuntimeError('Source identity mismatch')
env=dict(os.environ,CUDA_VISIBLE_DEVICES='',OMP_NUM_THREADS='1',MKL_NUM_THREADS='1')
run(sys.executable,ROOT/'research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3c/moe-bridge'/'run.py','--mode','engineering','--output',OUTPUT,cwd=ROOT,env=env)
print('CPU synthetic engineering complete. Save Version retains:',OUTPUT)
print('No GPU, real assets, or Confirmation were used. No additional Notebook test suite.')
